# Chapter 1 — micrograd: backpropagation from nothing

From *Neural Networks: Zero to Hero — The Textbook*.

Run each cell with **Shift+Enter**. Before you run one, say out loud what you expect it to print; being wrong is the useful part.


## Chapter 1 — micrograd: backpropagation from nothing

**Video:** 2h25m · [youtu.be/VMj-3S1tku0](https://youtu.be/VMj-3S1tku0) · **Runs on:** any laptop, no GPU, no libraries at all.

### The problem

You have a network with knobs and a loss. You need the gradient: for every knob, how does the loss respond? Doing that by hand for 41 knobs is tedious. For 175 billion it is impossible. You need a machine that computes gradients automatically, and to trust it you have to build one.

> **Say it to a six-year-old.** You built a tower of blocks and it fell over. You want to know which block was the problem. So you go backwards from the top, asking each block "did you wobble because of the block under you?" and you keep asking down the tower until you find the one at fault. Then you fix that block a tiny bit. That is all this chapter is: going backwards down the tower.

### What it structurally is

**micrograd** is an **automatic differentiation engine**: about 100 lines of Python that record arithmetic as it happens, then run the chain rule backward through the recording. It operates on single numbers rather than tensors, which makes it slow and completely transparent.

### Step 1 — wrap numbers in an object

Instead of a bare `2.0`, create an object holding two fields: `data`, the number, and `grad`, the sensitivity of the final loss to this number, starting at 0.

**Run it.**

In [ ]:
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None   # what to do during the backward pass
        self._prev = set(_children)     # which Values produced this one
        self._op = _op                  # which operation produced it (for display)
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

a = Value(2.0)
print(a)

**What you should see:**

**Expected output:**

```
Value(data=2.0, grad=0.0)
```

[verified]

`grad` starts at 0 because before any backward pass, we have not yet learned that this number influences anything.

### Step 2 — make arithmetic record itself

Redefine `+` and `*` so the result remembers where it came from. In Python, defining `__add__` on a class is what makes the `+` symbol work on it.

**Run it.** (Add these methods inside the `Value` class.)

These go **inside the class above** — paste them into that cell rather than running this on its own. The finished class is assembled in a runnable cell at the end of this notebook.

```python
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad     # addition passes gradient through unchanged
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad   # multiplication swaps the values
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
```

Two things are happening in each method:

1. **Forward:** compute the result now, and record the parents in `_children`.
2. **Backward:** store a small function that, *when later called*, will push gradient from this node back to its parents using the local rules from section 1.7.

The `_backward` function is not run yet. It is stored for later, which is the trick that makes the whole thing work.

### Step 3 — the graph builds itself

Computing `d = a*b + c` now produces a graph: `a` and `b` feed a multiply node, whose output plus `c` feeds an add node, whose output is `d`. Nobody declared that graph. Running the code built it.

This is called **define-by-run**, and it is why PyTorch feels like ordinary Python rather than a separate language. [standard]

### Step 4 — walk the graph backward, in the right order

Before computing a node's gradient you must have finished every node it feeds into. Sorting a graph so that this always holds is a **topological sort**: repeatedly take nodes whose dependencies are already handled. It is the only computer science in the lecture.

**Run it.** (Also inside the class.)

These go **inside the class above** — paste them into that cell rather than running this on its own. The finished class is assembled in a runnable cell at the end of this notebook.

```python
    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0                  # the loss is perfectly sensitive to itself
        for node in reversed(topo):
            node._backward()
```

`self.grad = 1.0` is the seed of the whole process. Nudging the loss by 0.001 changes the loss by 0.001, a sensitivity of exactly 1. Every other gradient in the network is that 1 propagated backward and multiplied by local rules.

**Run it.** The full test:

In [ ]:
a = Value(2.0); b = Value(-3.0); c = Value(10.0)
d = a*b + c
d.backward()
print("d =", d.data, "| a.grad =", a.grad, "| b.grad =", b.grad, "| c.grad =", c.grad)

**What you should see:**

**Expected output:**

```
d = 4.0 | a.grad = -3.0 | b.grad = 2.0 | c.grad = 1.0
```

[verified]

**Check every one of those by hand**, because if you follow this you have understood backpropagation:

- `a.grad = -3.0`: `a` is multiplied by `b`, so nudging `a` up by 1 changes the product by `b`, which is −3. The add passes that through unchanged.
- `b.grad = 2.0`: symmetrically, `a`'s value.
- `c.grad = 1.0`: `c` only feeds an addition, which passes sensitivity through untouched.

You can confirm all three by nudging, exactly as in section 1.5. Do it once.

### Step 5 — the accumulation bug that everyone hits

If a variable is used twice, its gradients must **accumulate** (`+=`), not overwrite (`=`). Using `b` in two places means it influences the loss through two separate routes, and the total influence is their sum. Overwriting silently discards one route.

**Run it.** See the bug and the fix:

In [ ]:
a = Value(3.0)
b = a + a          # a is used twice
b.backward()
print("a.grad =", a.grad, "(correct answer is 2.0: db/da = 1 + 1)")

**What you should see:**

**Expected output:**

```
a.grad = 2.0 (correct answer is 2.0: db/da = 1 + 1)
```

[verified]

With `=` instead of `+=` this prints 1.0, which is wrong, and nothing warns you. This is also why PyTorch makes you call `zero_grad()` before every step: gradients accumulate by design, so you must clear them yourself, and forgetting to is one of the most common bugs in the field. [standard]

### Step 6 — add tanh, and the rest of the operations

**Run it.** (`tanh` needs `import math` at the top of your file. The rest go inside the class.)

These go **inside the class above** — paste them into that cell rather than running this on its own. The finished class is assembled in a runnable cell at the end of this notebook.

```python
    def __pow__(self, other):
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out

    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad     # the slope of tanh, from section 1.10
        out._backward = _backward
        return out

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other       # lets sum() work
    def __rmul__(self, other): return self * other       # lets 2 * value work
    def __truediv__(self, other): return self * other**-1
```

Notice how few real rules there are. Subtraction is addition with a negation. Division is multiplication by a power of −1. Only `+`, `*`, `**`, and `tanh` need genuine derivative rules; everything else is composition. **That is the deep point of the chapter**: an arbitrarily complicated function needs only a handful of local rules, because the chain rule assembles the rest.

The `tanh` backward rule, `1 - t²`, is the slope you printed in section 1.10. At `t = ±0.999`, that is `1 − 0.998 = 0.0013`, near zero. Remember this number in Chapter 4.

### Step 7 — build a neural network on top

**Run it.**

In [ ]:
import random

class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(0.0)
    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)   # weighted sum + bias
        return act.tanh()                                        # squash
    def parameters(self):
        return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    def parameters(self):
        return [p for l in self.layers for p in l.parameters()]

random.seed(1337)
n = MLP(3, [4, 4, 1])      # 3 inputs -> 4 neurons -> 4 neurons -> 1 output
print("number of parameters:", len(n.parameters()))

**What you should see:**

**Expected output:**

```
number of parameters: 41
```

[verified] [transcript]

**Where 41 comes from**, so no number in this book is unexplained: layer 1 has 4 neurons each with 3 weights and 1 bias, so 4×(3+1) = 16. Layer 2 has 4 neurons each with 4 weights and a bias, 4×(4+1) = 20. Layer 3 has 1 neuron with 4 weights and a bias, 5. Total 16+20+5 = **41**.

### Step 8 — train it

Four training examples, each 3 numbers, with desired outputs +1 or −1.

**Run it.**

In [ ]:
xs = [[2.0, 3.0, -1.0],
      [3.0, -1.0, 0.5],
      [0.5, 1.0, 1.0],
      [1.0, 1.0, -1.0]]
ys = [1.0, -1.0, -1.0, 1.0]

for k in range(21):
    ypred = [n(x) for x in xs]                                   # forward pass
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))  # squared error
    for p in n.parameters():
        p.grad = 0.0                                             # RESET the gradients
    loss.backward()                                              # backward pass
    for p in n.parameters():
        p.data += -0.05 * p.grad                                 # step downhill
    if k % 5 == 0 or k == 20:
        print(f"step {k:2d}  loss {loss.data:.6f}")

print("final predictions:", [round(y.data, 4) for y in [n(x) for x in xs]])

**What you should see:**

**Expected output:**

```
step  0  loss 3.140718
step  5  loss 0.140404
step 10  loss 0.069491
step 15  loss 0.045248
step 20  loss 0.033226
final predictions: [0.9144, -0.9072, -0.9254, 0.9]
```

[verified]

Targets were `[1, −1, −1, 1]` and the network now outputs `[0.91, −0.91, −0.93, 0.90]`. That is the entire loop, and every neural network ever trained is this loop with bigger tensors.

**The minus sign is the whole algorithm.** `p.data += -0.05 * p.grad`. The gradient says which way makes the loss *bigger*, so you step the other way. Flip that minus to a plus and the loss will climb instead, which is worth doing once just to watch it happen.

**Why 0.05.** That is the **learning rate**, chosen by trial. Too small and the loss crawls; too big and it overshoots and diverges. Try 0.5 and 0.001 and watch both failure modes. Chapter 3 shows how to choose it properly rather than guessing.

### Distinctive features

- **Define-by-run**: the graph is built by executing the code, not declared in advance.
- **Scalars, not tensors**: real frameworks do this on whole arrays for speed. The math is identical; only the bookkeeping is uglier.
- **A deliberately PyTorch-shaped API**: late in the lecture Karpathy writes the same example in real PyTorch and gets matching numbers, which retroactively demystifies the real library.

**Run it.** The same gradients, from PyTorch:

In [ ]:
import torch
a = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([-3.0], requires_grad=True)
c = torch.tensor([10.0], requires_grad=True)
d = a*b + c
d.backward()
print(a.grad.item(), b.grad.item(), c.grad.item())

**What you should see:**

**Expected output:**

```
-3.0 2.0 1.0
```

[verified]

Identical to your 100-line engine. PyTorch is this, plus tensors, plus GPU kernels, plus twenty years of engineering.

### Exercises

1. **Add an `exp()` method** with the correct backward rule. Hint: the derivative of `eˣ` is `eˣ`, so `self.grad += out.data * out.grad`. Verify by nudging.
2. **Break the accumulation.** Change `+=` to `=` in `__add__`, rerun `b = a + a`, and confirm you get 1.0 instead of 2.0.
3. **Remove the nonlinearity.** Delete `.tanh()` from `Neuron.__call__` and retrain. The loss will still fall (this problem is nearly linear), but check with section 1.10's argument why depth is now pointless.
4. **Gradient check.** Pick any parameter, nudge its `data` by `h = 1e-5`, recompute the loss, and compare `(loss_new − loss_old)/h` with the `.grad` your engine reported. They should agree to about four decimal places. This is how every autodiff library is tested in practice. [standard]

### Troubleshooting

| Symptom | Cause |
|---|---|
| `TypeError: unsupported operand type(s) for +: 'int' and 'Value'` | `sum()` starts at integer 0; you need `__radd__` |
| Loss goes up, not down | Missing minus sign in the update, or learning rate far too large |
| Loss stuck exactly the same | You forgot `loss.backward()`, or all grads are 0 because you reset them after backward instead of before |
| Loss becomes `nan` | Learning rate too high; parameters exploded to infinity. Lower it to 0.01 |
| `RecursionError` in `build` | Your graph got very deep; raise the limit with `sys.setrecursionlimit(10000)` |

> **For the PhD in the room.** This is textbook reverse-mode AD over a dynamically constructed DAG (directed acyclic graph, meaning arrows never form a loop), with each primitive registering a vector-Jacobian product and a topological order guaranteeing that a node's adjoint is complete before it is used. The accumulation with `+=` is what makes fan-out correct, corresponding to the multivariable chain rule summing over all paths. Two things it omits that production systems care about: checkpointing, meaning recomputing activations rather than storing them, to trade compute for memory, and any notion of higher-order derivatives, which would require the backward pass itself to be differentiable, that is, built out of `Value` operations rather than raw floats. Karpathy's later `micrograd` variants and JAX's design differ exactly here.

### 30-second version

Every calculation a network performs is a chain of tiny operations. Each operation knows how sensitive its output is to its inputs, which for addition is "pass it through" and for multiplication is "swap the values." Multiply those sensitivities backward along the chain and you learn how every knob in the network affects the final error. That is backpropagation, it fits in 100 lines of Python, and the version inside PyTorch differs only by working on whole arrays at once.

---

## The finished engine, assembled

Every fragment above, in one runnable cell. Run this and you have a working autograd engine and a trained neural network.

In [ ]:
import math, random

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op
    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    def __pow__(self, other):
        out = Value(self.data ** other, (self,), f'**{other}')
        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward
        return out
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) - 1) / (math.exp(2*x) + 1)
        out = Value(t, (self,), 'tanh')
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        return out
    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other
    def __truediv__(self, other): return self * other**-1
    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()

# --- demo 1: numerical derivative
def f(x): return 3*x**2 - 4*x + 5
h = 0.001; x = 3.0
print("f(3.0) =", f(3.0))
print("numerical derivative at x=3:", (f(x+h) - f(x)) / h)

# --- demo 2: a tiny graph
a = Value(2.0); b = Value(-3.0); c = Value(10.0)
d = a*b + c
d.backward()
print("d =", d.data, "| a.grad =", a.grad, "| b.grad =", b.grad, "| c.grad =", c.grad)

# --- demo 3: the MLP
class Neuron:
    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(0.0)
    def __call__(self, x):
        act = sum((wi*xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh()
    def parameters(self): return self.w + [self.b]

class Layer:
    def __init__(self, nin, nout): self.neurons = [Neuron(nin) for _ in range(nout)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    def parameters(self): return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]
    def __call__(self, x):
        for layer in self.layers: x = layer(x)
        return x
    def parameters(self): return [p for l in self.layers for p in l.parameters()]

random.seed(1337)
n = MLP(3, [4, 4, 1])
print("number of parameters:", len(n.parameters()))

xs = [[2.0,3.0,-1.0],[3.0,-1.0,0.5],[0.5,1.0,1.0],[1.0,1.0,-1.0]]
ys = [1.0, -1.0, -1.0, 1.0]

for k in range(21):
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt, yout in zip(ys, ypred))
    for p in n.parameters(): p.grad = 0.0
    loss.backward()
    for p in n.parameters(): p.data += -0.05 * p.grad
    if k % 5 == 0 or k == 20:
        print(f"step {k:2d}  loss {loss.data:.6f}")
print("final predictions:", [round(y.data, 4) for y in [n(x) for x in xs]])